# Notebook 3 of 3 - Traces and Evaluation
Owner: Quang Tran - PM

---

### PaySprint AI Investment Agent Workflow

| Step | Notebook | Owner | Output |
|------|----------|-------|--------|
| 1 | `data_pipeline` | Reema Eid (Data Engineer) | Validated + ranked stock universe → `data/screener_override.json`, `data/master_stock_data.csv` |
| 2 | `agent_definition` | Hyunju Yu (AI Engineer) | ReAct agent traces → `data/traces/trace_1.json` … `trace_5b.json` |
| **3** | **`traces_evaluation` (this notebook)** | **Quang Tran (PM)** | **Loads traces from Notebook 2 → LLM judge scores, backtesting, consistency → `data/evaluation_results.csv`** |

---

### What this notebook does

This notebook picks up where Notebooks 1 and 2 left off:
- **Step 0**: Loads the pipeline-validated stock universe from Notebook 1 (`master_stock_data.csv`) for context
- **Traces 1–5**: Loads the agent traces saved by Notebook 2 (`data/traces/trace_*.json`); runs the agent only as a fallback if a trace file is missing
- **LLM Judge**: Evaluates each report on 4 dimensions (reasoning, risk alignment, clarity, plan accuracy)
- **Backtesting**: Compares the conservative portfolio against SPY over 63 trading days
- **Consistency**: Measures pick stability across 3 repeated runs of the same profile

In [1]:
# === Setup: load BOTH providers ============================================
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'])

from dotenv import load_dotenv
load_dotenv()

import os, json, threading
sys.path.insert(0, '.')

from openai import OpenAI
import paysprint_agent as core
from paysprint_agent import (
    run_agent, test_rejection, llm_judge,
    save_trace, load_trace, print_trace_summary,
    estimate_cost, backtest, consistency_test,
)
import pandas as pd

GEMINI_BASE_URL = 'https://generativelanguage.googleapis.com/v1beta/openai/'
GEMINI_KEY = os.getenv('GEMINI_API_KEY', '')
OPENAI_KEY = os.getenv('OPENAI_API_KEY', '')

print('API keys detected:')
print(f'  GEMINI_API_KEY : {"FOUND" if GEMINI_KEY else "NOT FOUND"}')
print(f'  OPENAI_API_KEY : {"FOUND" if OPENAI_KEY else "NOT FOUND"}')

if not GEMINI_KEY and not OPENAI_KEY:
    raise EnvironmentError('No API keys found. Set GEMINI_API_KEY and/or OPENAI_API_KEY.')

_provider_local = threading.local()

def _dual_client(model=None):
    """Route to Gemini or OpenAI based on thread-local provider override."""
    provider = getattr(_provider_local, 'provider', 'gemini' if GEMINI_KEY else 'openai')
    if provider == 'gemini':
        if not GEMINI_KEY:
            raise EnvironmentError('GEMINI_API_KEY not set.')
        return OpenAI(api_key=GEMINI_KEY, base_url=GEMINI_BASE_URL)
    if not OPENAI_KEY:
        raise EnvironmentError('OPENAI_API_KEY not set.')
    return OpenAI(api_key=OPENAI_KEY)

core._get_client = _dual_client
print('\nDual-provider client ready.')

API keys detected:
  GEMINI_API_KEY : FOUND
  OPENAI_API_KEY : FOUND

Dual-provider client ready.


In [2]:
# === Model selection =========================================================
if GEMINI_KEY and OPENAI_KEY:
    MODEL_1 = os.getenv('MODEL_REASONING', 'gemini-2.5-flash')   # Gemini  main traces
    MODEL_2 = os.getenv('MODEL_SUMMARY',   'gpt-4o-mini')        # OpenAI  comparison
elif GEMINI_KEY:
    MODEL_1 = os.getenv('MODEL_REASONING', 'gemini-2.5-flash')
    MODEL_2 = os.getenv('MODEL_SUMMARY',   'gemini-2.5-flash-lite')
else:
    MODEL_1 = os.getenv('MODEL_REASONING', 'gpt-4o')
    MODEL_2 = os.getenv('MODEL_SUMMARY',   'gpt-4o-mini')

core.MODEL_REASONING = MODEL_1
core.MODEL_SUMMARY   = MODEL_2

# Set main-thread default: Traces 1-4 all use MODEL_1
PROVIDER_1 = 'gemini' if MODEL_1.startswith('gemini') else 'openai'
PROVIDER_2 = 'gemini' if MODEL_2.startswith('gemini') else 'openai'
_provider_local.provider = PROVIDER_1

print(f'Model 1 {PROVIDER_1.upper()} : {MODEL_1}')
print(f'Model 2 {PROVIDER_2.upper()} : {MODEL_2}')
print(f'Main thread provider set to: {PROVIDER_1}')
print()
print('Pricing reference (per 1M tokens):')
for model, prices in core.MODEL_PRICES.items():
    print(f'  {model}: ${prices["input"]}/1M input, ${prices["output"]}/1M output')

Model 1 GEMINI : gemini-2.5-flash
Model 2 OPENAI : gpt-4o-mini
Main thread provider set to: gemini

Pricing reference (per 1M tokens):
  gpt-4o: $2.5/1M input, $10.0/1M output
  gpt-4o-mini: $0.15/1M input, $0.6/1M output
  gemini-2.5-flash: $0.3/1M input, $2.5/1M output
  gemini-2.5-flash-lite: $0.1/1M input, $0.4/1M output


---
## Step 0 Pipeline Context from Notebook 1

Before running agent traces, we load the scoring output from Notebook 1 (data_pipeline)
to see what the data pipeline ranked across all candidate stocks.

This gives a reference point for the trace analysis: we can check whether the agent
consistently picks the stocks the pipeline scored highest, or whether the LLM reasoning
diverges from the quantitative signal.

In [3]:
# === Load pipeline data from Notebook 1 for context =========================
# master_stock_data.csv contains all candidate tickers scored by trend, sentiment,
# and fundamentals. We load it here to show what the data pipeline found, and
# later compare it against what the agent actually picked in each trace.
import os, pandas as pd

pipeline_path = 'data/master_stock_data.csv'
if os.path.exists(pipeline_path):
    pipeline_df = pd.read_csv(pipeline_path)
    print(f'Pipeline data loaded: {len(pipeline_df)} stocks scored by Notebook 1 (data_pipeline)')

    # Show the top-ranked stocks with key signals
    show_cols = [c for c in ['ticker', 'trend_score', 'avg_sentiment', 'slope_per_day',
                              'revenue_growth', 'risk_label', 'suggested_investment_amount',
                              'used_example_data'] if c in pipeline_df.columns]
    if show_cols and 'trend_score' in pipeline_df.columns:
        print('\nTop 10 pipeline-ranked stocks (by trend score):')
        print(pipeline_df[show_cols].sort_values('trend_score', ascending=False).head(10).to_string(index=False))
    print()
    print('These are the candidates the agent will research across the 5 traces below.')
else:
    pipeline_df = None
    print('No pipeline data found  run Notebook 1 (data_pipeline) first.')
    print('Continuing without pipeline context.')

Pipeline data loaded: 12 stocks scored by Notebook 1 (data_pipeline)

Top 10 pipeline-ranked stocks (by trend score):
ticker  trend_score  avg_sentiment  slope_per_day  revenue_growth risk_label  suggested_investment_amount  used_example_data
  NVDA        0.822         -0.013         0.3002           0.852      Lower                       1020.0              False
  AVGO        0.773          0.031         0.9241           0.479     Medium                        700.0              False
  MRVL        0.754          0.214         1.6186           0.276       High                        370.0              False
  INTC        0.741          0.169         0.7666           0.072       High                        370.0              False
  ORCL        0.741          0.105         0.2126           0.206       High                        370.0              False
    MU        0.728          0.027         5.3422           1.963       High                        370.0              False
  SNDK 

---
## The 5 Traces

| Trace | Scenario | Model | Purpose |
|-------|----------|-------|---------|
| 1 | Conservative investor, $5,000, 12 months | MODEL_1 | Normal run, low-risk strategy |
| 2 | Aggressive investor, $10,000, 6 months | MODEL_1 | High-risk strategy |
| 3 | Moderate investor, $3,000, 24 months (holds AAPL) | MODEL_1 | Balanced, existing holdings |
| 4 | Off-topic rejection tests | MODEL_1 | Graceful rejection |
| 5 | Same profile as Trace 1, MODEL_1 vs MODEL_2 | Both | LLM comparison |

---
## Trace 1 - Conservative Investor ($5,000, 12 months)

In [4]:
import os

PROFILE_1 = {
    'name':              'Conservative Carol',
    'budget':            5000.0,
    'aggressiveness':    'conservative',
    'horizon_months':    12,
    'current_holdings':  {},
    'preferred_sectors': [],
}

trace1_path = 'data/traces/trace_1.json'
if os.path.exists(trace1_path):
    trace1 = load_trace(trace_id=1)
    print('Loaded Trace 1 from data/traces/trace_1.json (produced by Notebook 2).')
else:
    print('trace_1.json not found — run Notebook 2 (agent_definition) first to generate traces.')
    print('Running agent now as fallback...\n')
    trace1 = run_agent(PROFILE_1, model=MODEL_1, max_turns=20, verbose=True)
    trace1.update({'trace_id': 1, 'label': f'Conservative $5k 12mo [{MODEL_1}]', 'profile': PROFILE_1})
    save_trace(trace1, trace_id=1)

print('\n' + '=' * 60)
print('FINAL REPORT:')
print('=' * 60)
print(trace1.get('report', ''))
print_trace_summary(trace1)

Loaded Trace 1 from data/traces/trace_1.json (produced by Notebook 2).

FINAL REPORT:
COMPLIANCE NOTICE: This information is for informational purposes only and does NOT constitute financial advice. PaySprint is NOT a registered investment advisor. Please consult a licensed financial advisor before making any investment decisions.

Here is your personalized investment plan, Conservative Carol:

We've identified two strong candidates that align with your conservative investment strategy and 12-month horizon. These companies demonstrate stable fundamentals, positive market sentiment, and favorable technical indicators.

**Selected Stocks:**

*   **Johnson & Johnson (JNJ):** JNJ is a healthcare giant known for its pharmaceutical, medical devices, and consumer health products. It exhibits strong fundamentals with a solid EPS and revenue growth, coupled with a manageable debt-to-equity ratio. Recent news indicates positive sentiment, driven by strategic investments and acquisitions. The sto

---
## Trace 2 - Aggressive Investor ($10,000, 6 months)

In [5]:
PROFILE_2 = {
    'name':              'Aggressive Alex',
    'budget':            10000.0,
    'aggressiveness':    'aggressive',
    'horizon_months':    6,
    'current_holdings':  {},
    'preferred_sectors': ['Technology'],
}

trace2_path = 'data/traces/trace_2.json'
if os.path.exists(trace2_path):
    trace2 = load_trace(trace_id=2)
    print('Loaded Trace 2 from data/traces/trace_2.json (produced by Notebook 2).')
else:
    print('trace_2.json not found — run Notebook 2 (agent_definition) first to generate traces.')
    print('Running agent now as fallback...\n')
    trace2 = run_agent(PROFILE_2, model=MODEL_1, max_turns=20, verbose=True)
    trace2.update({'trace_id': 2, 'label': f'Aggressive $10k 6mo [{MODEL_1}]', 'profile': PROFILE_2})
    save_trace(trace2, trace_id=2)

print('\n' + '=' * 60)
print('FINAL REPORT:')
print('=' * 60)
print(trace2.get('report', ''))
print_trace_summary(trace2)

Loaded Trace 2 from data/traces/trace_2.json (produced by Notebook 2).

FINAL REPORT:
| Ticker | Weight | Allocation (USD) | Est. Shares |
|:-------|:-------|:-----------------|:------------|
| AMD    | 0.55   | 5500.0           | 10          |
| NVDA   | 0.45   | 4500.0           | 21          |

**Risk Disclosure:**

Investing in the stock market involves inherent risks, and the value of investments can fluctuate. Market risk, including general economic downturns or specific sector-related challenges, could negatively impact the performance of your portfolio. The data used for this analysis is based on historical information and current market conditions, which may not be indicative of future results. There is no guarantee of returns, and you could lose money, including your principal investment. It is crucial to understand that this report is for informational purposes only and does not constitute financial advice. Before making any investment decisions, you MUST consult with a lice

---
## Trace 3 - Moderate Investor ($3,000, 24 months, already holds AAPL)

In [6]:
PROFILE_3 = {
    'name':              'Moderate Mike',
    'budget':            3000.0,
    'aggressiveness':    'moderate',
    'horizon_months':    24,
    'current_holdings':  {'AAPL': 3},
    'preferred_sectors': [],
}

trace3_path = 'data/traces/trace_3.json'
if os.path.exists(trace3_path):
    trace3 = load_trace(trace_id=3)
    print('Loaded Trace 3 from data/traces/trace_3.json (produced by Notebook 2).')
else:
    print('trace_3.json not found — run Notebook 2 (agent_definition) first to generate traces.')
    print('Running agent now as fallback...\n')
    trace3 = run_agent(PROFILE_3, model=MODEL_1, max_turns=20, verbose=True)
    trace3.update({'trace_id': 3, 'label': f'Moderate $3k 24mo [{MODEL_1}]', 'profile': PROFILE_3})
    save_trace(trace3, trace_id=3)

print('\n' + '=' * 60)
print('FINAL REPORT:')
print('=' * 60)
print(trace3.get('report', ''))
print_trace_summary(trace3)

Loaded Trace 3 from data/traces/trace_3.json (produced by Notebook 2).

FINAL REPORT:
COMPLIANCE NOTICE: This report is for INFORMATIONAL PURPOSES ONLY and does NOT constitute financial advice, a solicitation, or an offer to buy or sell any security. Please consult a licensed financial advisor before making any investment decisions.

Here is your personalized investment plan based on a moderate strategy and a 24-month horizon:

**Selected Stocks:**

*   **Alphabet (GOOGL)**: Google's parent company, Alphabet, shows strong positive price momentum with a significant 12-month forecast. Its robust revenue growth and healthy free cash flow indicate a strong underlying business.
    *   **Reasoning**: GOOGL was selected due to its leading positive price momentum (slope_per_day: 0.6077, 12-month forecast: $521.17), strong revenue growth (0.218), and a reasonable P/E ratio for a growth stock (28.09). This aligns with a moderate strategy that weights slightly towards momentum.

*   **Apple (AAP

---
## Trace 4 - Graceful Rejection (2 examples)

The agent must refuse irrelevant questions without calling any tools.

In [7]:
rejection_inputs = [
    'What is the capital of France?',
    'Can you write me a Python script to sort a list?',
]

trace4_path = 'data/traces/trace_4_rejection.json'
if os.path.exists(trace4_path):
    saved4 = load_trace(trace_id='4_rejection')
    rejection_results = saved4.get('rejection_tests', [])
    print('Loaded Trace 4 from data/traces/trace_4_rejection.json (produced by Notebook 2).\n')
    for i, r in enumerate(rejection_results, 1):
        status = 'PASS (no tools called)' if not r['tool_calls_made'] else 'FAIL (tools were called!)'
        print(f'Rejection {i}: {status}')
        print(f'  Input:    "{r["user_message"]}"')
        print(f'  Response: {r["response"][:300]}')
        print()
else:
    print('trace_4_rejection.json not found — run Notebook 2 first.')
    print('Running rejection tests now as fallback...\n')
    rejection_results = []
    for i, msg in enumerate(rejection_inputs, 1):
        r = test_rejection(msg, model=MODEL_1)
        rejection_results.append(r)
        status = 'PASS (no tools called)' if not r['tool_calls_made'] else 'FAIL (tools were called!)'
        print(f'Rejection {i}: {status}')
        print(f'  Input:    "{msg}"')
        print(f'  Response: {r["response"][:300]}')
        print()
    save_trace({'rejection_tests': rejection_results, 'trace_id': '4_rejection'}, trace_id='4_rejection')

print('Trace 4 complete.')

Loaded Trace 4 from data/traces/trace_4_rejection.json (produced by Notebook 2).

Rejection 1: PASS (no tools called)
  Input:    "What is the capital of France?"
  Response: I'm specialized in investment research. I can't help with that, but I'm happy to research stocks or build an investment plan for you.

Rejection 2: PASS (no tools called)
  Input:    "Can you write me a Python script to sort a list?"
  Response: I'm specialized in investment research. I can't help with that, but I'm happy to research stocks or build an investment plan for you.

Trace 4 complete.


---
## Trace 5 - LLM Comparison (same profile, two models)

The same investor profile runs through both models so we can compare quality and cost.

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed

COMPARE_PROFILE = PROFILE_1.copy()
COMPARE_PROFILE['name'] = 'Compare Investor'

trace5a_path = 'data/traces/trace_5a.json'
trace5b_path = 'data/traces/trace_5b.json'

if os.path.exists(trace5a_path) and os.path.exists(trace5b_path):
    trace5a = load_trace(trace_id='5a')
    trace5b = load_trace(trace_id='5b')
    print('Loaded Trace 5a and 5b from data/traces/ (produced by Notebook 2).')
else:
    print('trace_5a/5b.json not found — run Notebook 2 first.')
    print('Running both models IN PARALLEL as fallback...')
    print(f'  Model 1: {MODEL_1}  ({PROVIDER_1.upper()})')
    print(f'  Model 2: {MODEL_2}  ({PROVIDER_2.upper()})')
    print('-' * 60)

    def run_model(label, model, provider):
        _provider_local.provider = provider
        print(f'[{label}] started on {provider.upper()}...')
        result = run_agent(COMPARE_PROFILE, model=model, max_turns=20, verbose=False)
        tokens = result.get('usage', {}).get('total_tokens', 0)
        print(f'[{label}] done  ({result.get("turns")} turns, {tokens:,} tokens)')
        return label, result

    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = {
            pool.submit(run_model, 'Model 1', MODEL_1, PROVIDER_1): 'Model 1',
            pool.submit(run_model, 'Model 2', MODEL_2, PROVIDER_2): 'Model 2',
        }
        results_map = {}
        for future in as_completed(futures):
            label, result = future.result()
            results_map[label] = result

    trace5a = results_map['Model 1']
    trace5b = results_map['Model 2']
    trace5a.update({'trace_id': '5a', 'label': f'Compare [{MODEL_1}]', 'profile': COMPARE_PROFILE})
    trace5b.update({'trace_id': '5b', 'label': f'Compare [{MODEL_2}]', 'profile': COMPARE_PROFILE})
    save_trace(trace5a, trace_id='5a')
    save_trace(trace5b, trace_id='5b')
    print('\nBoth traces saved.')

Loaded Trace 5a and 5b from data/traces/ (produced by Notebook 2).


In [9]:
# === Side-by-side comparison =================================================
cost5a = estimate_cost(trace5a)
cost5b = estimate_cost(trace5b)

W = 50  # column width

def col(val, width=W):
    return str(val)[:width].ljust(width)

# Pre-format values to avoid f-string nesting with backslashes
tot_a   = f'{cost5a["input_tokens"] + cost5a["output_tokens"]:,}'
tot_b   = f'{cost5b["input_tokens"] + cost5b["output_tokens"]:,}'
inp_a   = f'{cost5a["input_tokens"]:,}'
inp_b   = f'{cost5b["input_tokens"]:,}'
out_a   = f'{cost5a["output_tokens"]:,}'
out_b   = f'{cost5b["output_tokens"]:,}'
cost_a  = f'${cost5a["cost_usd"]:.6f}'
cost_b  = f'${cost5b["cost_usd"]:.6f}'
label_a = f'Model 1: {MODEL_1} ({PROVIDER_1.upper()})'
label_b = f'Model 2: {MODEL_2} ({PROVIDER_2.upper()})'

print('=' * 120)
print('  SIDE-BY-SIDE COMPARISON')
print(f'  Profile: {COMPARE_PROFILE["name"]} | ${COMPARE_PROFILE["budget"]:,.0f} | {COMPARE_PROFILE["aggressiveness"]} | {COMPARE_PROFILE["horizon_months"]}mo')
print('=' * 120)

# --- Stats table ---
print(f'\n  {"Metric":<22} {col(label_a)} {col(label_b)}')
print(f'  {"-"*22} {"-"*W} {"-"*W}')
print(f'  {"Turns":<22} {col(trace5a.get("turns"))} {col(trace5b.get("turns"))}')
print(f'  {"Total tokens":<22} {col(tot_a)} {col(tot_b)}')
print(f'  {"Input tokens":<22} {col(inp_a)} {col(inp_b)}')
print(f'  {"Output tokens":<22} {col(out_a)} {col(out_b)}')
print(f'  {"Est. cost (USD)":<22} {col(cost_a)} {col(cost_b)}')

# --- Tools called ---
tools5a = [f'Turn {tc["turn"]}: {tc["tool"]}' for tc in trace5a.get('tool_calls', [])]
tools5b = [f'Turn {tc["turn"]}: {tc["tool"]}' for tc in trace5b.get('tool_calls', [])]
print(f'\n  {"TOOLS CALLED":<22} {col("Model 1")} {col("Model 2")}')
print(f'  {"-"*22} {"-"*W} {"-"*W}')
for i in range(max(len(tools5a), len(tools5b))):
    a = tools5a[i] if i < len(tools5a) else ''
    b = tools5b[i] if i < len(tools5b) else ''
    print(f'  {"":<22} {col(a)} {col(b)}')

# --- Reports ---
print(f'\n{"=" * 120}')
print('  FINAL REPORTS')
print('=' * 120)

print(f'\n  {label_a}')
print(f'  {"-" * 60}')
for line in trace5a.get('report', '').strip().split('\n'):
    print(f'  {line}')

print(f'\n  {label_b}')
print(f'  {"-" * 60}')
for line in trace5b.get('report', '').strip().split('\n'):
    print(f'  {line}')

  SIDE-BY-SIDE COMPARISON
  Profile: Compare Investor | $5,000 | conservative | 12mo

  Metric                 Model 1: gemini-2.5-flash (GEMINI)                 Model 2: gpt-4o-mini (OPENAI)                     
  ---------------------- -------------------------------------------------- --------------------------------------------------
  Turns                  7                                                  6                                                 
  Total tokens           13,863                                             13,877                                            
  Input tokens           12,953                                             12,816                                            
  Output tokens          910                                                1,061                                             
  Est. cost (USD)        $0.006161                                          $0.002559                                         

  TOOLS CALLED          

---
## LLM Judge Evaluation

An LLM scores each report on 4 dimensions (1-5 scale):
- **Reasoning quality**: Is the analysis logical and grounded in tool data?
- **Risk alignment**: Do picks match the investor's stated strategy?
- **Clarity**: Easy to understand for a non-expert?
- **Plan accuracy**: Are the dollar allocations and share counts correct?

### Evaluation Thresholds (Deployment Gate)

The following minimum scores must be met for the agent to be considered production-ready. A trace that fails **any single threshold** is flagged for review.

| Dimension | Minimum score | Rationale |
|-----------|--------------|-----------|
| **Overall** | ≥ 4.0 / 5.0 | Below 4.0 indicates the report would not be useful to a retail investor |
| **Risk alignment** | ≥ 4.0 / 5.0 | Misalignment between picks and investor strategy is a safety failure - non-negotiable |
| **Plan accuracy** | ≥ 4.0 / 5.0 | Incorrect dollar allocations or share counts are a functional defect |
| **Reasoning quality** | ≥ 3.0 / 5.0 | Minimum bar for data-grounded analysis; below this the agent is guessing |
| **Clarity** | ≥ 3.0 / 5.0 | Report must be readable without a finance background |

In [10]:
import os
os.makedirs('data', exist_ok=True)

eval_runs = [
    (trace1,  PROFILE_1,       '1 - Conservative'),
    (trace2,  PROFILE_2,       '2 - Aggressive'),
    (trace3,  PROFILE_3,       '3 - Moderate'),
    (trace5a, COMPARE_PROFILE, f'5a - {MODEL_1}'),
    (trace5b, COMPARE_PROFILE, f'5b - {MODEL_2}'),
]

print(f'Running LLM judge with {MODEL_2} ({PROVIDER_2.upper()})...\n')

# Switch to PROVIDER_2 for judge calls, restore PROVIDER_1 when done
_provider_local.provider = PROVIDER_2
eval_records = []
try:
    for trace, profile, label in eval_runs:
        scores = llm_judge(trace['report'], profile)
        scores['trace']  = label
        scores['model']  = trace.get('model', '')
        scores['tokens'] = trace.get('usage', {}).get('total_tokens', 0)
        scores['turns']  = trace.get('turns', 0)
        eval_records.append(scores)
        overall   = scores.get('overall', 'N/A')
        strengths = scores.get('strengths', '')[:70]
        print(f'  Trace {label}: overall={overall}  | {strengths}')
finally:
    _provider_local.provider = PROVIDER_1  # restore main-thread provider

eval_df = pd.DataFrame(eval_records)
eval_df.to_csv('data/evaluation_results.csv', index=False)
print('\nSaved to data/evaluation_results.csv')

Running LLM judge with gpt-4o-mini (OPENAI)...

  Trace 1 - Conservative: overall=4.25  | The report effectively identifies two stable companies that align well
  Trace 2 - Aggressive: overall=4.25  | The report effectively aligns the stock selections with the user's agg
  Trace 3 - Moderate: overall=4.5  | The report provides a well-reasoned analysis of selected stocks based 
  Trace 5a - gemini-2.5-flash: overall=4.5  | The report provides a well-reasoned selection of stocks that align wit
  Trace 5b - gpt-4o-mini: overall=4.5  | The report provides a well-reasoned selection of stocks that align wit

Saved to data/evaluation_results.csv


In [11]:
score_cols = ['trace', 'model', 'reasoning_quality', 'risk_alignment', 'clarity', 'plan_accuracy', 'overall', 'tokens', 'turns']
available  = [c for c in score_cols if c in eval_df.columns]
print('=== LLM Judge Scores ===')
print(eval_df[available].to_string(index=False))
print(f'\nMean overall score: {eval_df["overall"].mean():.2f} / 5.0')

=== LLM Judge Scores ===
                trace            model  reasoning_quality  risk_alignment  clarity  plan_accuracy  overall  tokens  turns
     1 - Conservative gemini-2.5-flash                  4               5        4              4     4.25    8100      5
       2 - Aggressive gemini-2.5-flash                  4               5        3              5     4.25   13648      7
         3 - Moderate gemini-2.5-flash                  5               4        4              5     4.50   17254      8
5a - gemini-2.5-flash gemini-2.5-flash                  4               5        4              5     4.50   13863      7
     5b - gpt-4o-mini      gpt-4o-mini                  4               5        4              5     4.50   13877      6

Mean overall score: 4.40 / 5.0


In [12]:
# === Deployment Threshold Gate ================================================
# Checks every trace against the minimum acceptable scores defined above.
# Any dimension that falls below its threshold is flagged FAIL.

THRESHOLDS = {
    'overall':           4.0,
    'risk_alignment':    4.0,
    'plan_accuracy':     4.0,
    'reasoning_quality': 3.0,
    'clarity':           3.0,
}

print('=== DEPLOYMENT THRESHOLD GATE ===\n')
print(f'{"Trace":<25} {"Dimension":<22} {"Score":>6}  {"Min":>5}  {"Result"}')
print('-' * 72)

all_pass = True
for _, row in eval_df.iterrows():
    label = str(row.get('trace', ''))
    for dim, min_score in THRESHOLDS.items():
        score = row.get(dim)
        if score is None:
            continue
        passed = float(score) >= min_score
        status = 'PASS' if passed else 'FAIL <<<'
        if not passed:
            all_pass = False
        print(f'{label:<25} {dim:<22} {float(score):>6.2f}  {min_score:>5.1f}  {status}')
    print()

print('=' * 72)
if all_pass:
    print('GATE RESULT: ALL TRACES PASS -- agent meets minimum quality thresholds.')
else:
    print('GATE RESULT: ONE OR MORE TRACES FAILED -- review flagged dimensions before deployment.')

=== DEPLOYMENT THRESHOLD GATE ===

Trace                     Dimension               Score    Min  Result
------------------------------------------------------------------------
1 - Conservative          overall                  4.25    4.0  PASS
1 - Conservative          risk_alignment           5.00    4.0  PASS
1 - Conservative          plan_accuracy            4.00    4.0  PASS
1 - Conservative          reasoning_quality        4.00    3.0  PASS
1 - Conservative          clarity                  4.00    3.0  PASS

2 - Aggressive            overall                  4.25    4.0  PASS
2 - Aggressive            risk_alignment           5.00    4.0  PASS
2 - Aggressive            plan_accuracy            5.00    4.0  PASS
2 - Aggressive            reasoning_quality        4.00    3.0  PASS
2 - Aggressive            clarity                  3.00    3.0  PASS

3 - Moderate              overall                  4.50    4.0  PASS
3 - Moderate              risk_alignment           4.00    4

---
## LLM Comparison - Cost and Quality (ROI Analysis)

In [13]:
cost5a = estimate_cost(trace5a)
cost5b = estimate_cost(trace5b)

score5a = next((r['overall'] for r in eval_records if '5a' in str(r.get('trace', ''))), 0)
score5b = next((r['overall'] for r in eval_records if '5b' in str(r.get('trace', ''))), 0)

cost_ratio   = cost5a['cost_usd'] / max(cost5b['cost_usd'], 0.0000001)
quality_diff = (score5a - score5b) / max(score5b, 0.001) * 100

print('=' * 60)
print('LLM COST AND QUALITY COMPARISON')
print('=' * 60)
print(f'\n{MODEL_1} (Model 1 - strong):')
print(f'  Tokens: {cost5a["input_tokens"]:,} input + {cost5a["output_tokens"]:,} output')
print(f'  Cost:   ${cost5a["cost_usd"]:.6f}')
print(f'  Score:  {score5a}/5')
print(f'\n{MODEL_2} (Model 2 - cheaper):')
print(f'  Tokens: {cost5b["input_tokens"]:,} input + {cost5b["output_tokens"]:,} output')
print(f'  Cost:   ${cost5b["cost_usd"]:.6f}')
print(f'  Score:  {score5b}/5')
print(f'\nROI Analysis:')
print(f'  {MODEL_1} costs {cost_ratio:.1f}x more than {MODEL_2}')
print(f'  Quality difference: {quality_diff:+.1f}% ({score5a} vs {score5b})')

if cost_ratio > 1:
    quality_per_dollar = (score5a - score5b) / (cost5a['cost_usd'] - cost5b['cost_usd'] + 0.0000001)
    print(f'  Quality gain per $0.001 extra: {quality_per_dollar * 0.001:.4f} points')

LLM COST AND QUALITY COMPARISON

gemini-2.5-flash (Model 1 - strong):
  Tokens: 12,953 input + 910 output
  Cost:   $0.006161
  Score:  4.5/5

gpt-4o-mini (Model 2 - cheaper):
  Tokens: 12,816 input + 1,061 output
  Cost:   $0.002559
  Score:  4.5/5

ROI Analysis:
  gemini-2.5-flash costs 2.4x more than gpt-4o-mini
  Quality difference: +0.0% (4.5 vs 4.5)
  Quality gain per $0.001 extra: 0.0000 points


---
## Backtesting

We compare what the agent would have recommended 3 months ago against what actually
happened to those stock prices, benchmarked against SPY (S&P 500 ETF).

### Backtesting Thresholds

| Metric | Threshold | Rationale |
|--------|-----------|-----------|
| Nominal portfolio return | > 0% | Portfolio must at minimum preserve capital over the test window |
| Alpha vs SPY | > −20% | Conservative strategies are expected to lag in bull markets; sustained underperformance beyond −20% signals strategy misalignment, not just market conditions |
| Worst single-stock loss | > −30% | An individual position declining more than 30% indicates unacceptable concentration risk for a conservative strategy |

**Scope caveat:** The 90-day window used here is a proof-of-concept to confirm the agent's picks are real, tradable securities with retrievable price history. A production evaluation requires a minimum 1–3 year window spanning at least one significant market drawdown to properly assess risk-adjusted returns.

In [14]:
from datetime import datetime, timedelta

# Use tickers the agent actually picked in Trace 1 (conservative profile)
# Extract from trace1 tool_calls so this stays dynamic
plan_calls = [tc for tc in trace1.get('tool_calls', []) if tc['tool'] == 'create_purchase_plan']
if plan_calls:
    backtest_tickers = [t for t in plan_calls[-1]['args'].get('tickers', []) if t != 'CASH'][:4]
else:
    backtest_tickers = ['JNJ', 'PG', 'KO', 'WMT']  # conservative fallback

backtest_date = (datetime.today() - timedelta(days=90)).strftime('%Y-%m-%d')

print(f'Tickers: {backtest_tickers}')
print(f'Backtesting from {backtest_date} (90 days ago) over 63 trading days...\n')

bt_df = backtest(backtest_tickers, backtest_date, horizon_days=63, benchmark='SPY')
print(bt_df.to_string(index=False))

stock_rows = bt_df[bt_df['ticker'] != 'SPY']
spy_rows   = bt_df[bt_df['ticker'] == 'SPY']

if 'return_pct' in stock_rows.columns and not stock_rows.empty:
    avg_return = stock_rows['return_pct'].mean()
    spy_val    = float(spy_rows['return_pct'].values[0]) if not spy_rows.empty else 0.0
    alpha      = avg_return - spy_val
    print(f'\nAverage portfolio return:  {avg_return:+.2f}%')
    print(f'SPY benchmark return:      {spy_val:+.2f}%')
    print(f'Alpha (excess return):     {alpha:+.2f}%')
    print(f'Verdict: {"OUTPERFORMED" if alpha > 0 else "UNDERPERFORMED"} the S&P 500 by {abs(alpha):.2f}%')

Tickers: ['JNJ', 'PG', 'KO', 'WMT']
Backtesting from 2026-03-21 (90 days ago) over 63 trading days...

ticker  entry_price  exit_price  return_pct  benchmark_return_pct  alpha_pct
   JNJ       234.07      228.39       -2.43                 14.23     -16.66
    PG       142.91      150.56        5.35                 14.23      -8.88
    KO        74.63       79.39        6.38                 14.23      -7.85
   WMT       120.49      117.18       -2.75                 14.23     -16.98
   SPY       653.70      746.74       14.23                 14.23       0.00

Average portfolio return:  +1.64%
SPY benchmark return:      +14.23%
Alpha (excess return):     -12.59%
Verdict: UNDERPERFORMED the S&P 500 by 12.59%


---
## Consistency Test

We run the same conservative profile 3 times and measure how often the agent recommends
the same stocks. Jaccard similarity: 1.0 = identical picks, 0.0 = completely different.

**Note:** This runs the agent 3 times — takes 3-5 minutes.

### Consistency Threshold

| Average Jaccard score | Verdict | Deployment implication |
|-----------------------|---------|----------------------|
| ≥ 0.70 | CONSISTENT | Acceptable for production use — agent picks reliably reflect the strongest signals |
| 0.40 – 0.69 | MODERATE | Acceptable for a prototype; root-cause the variance before moving to production |
| < 0.40 | INCONSISTENT | Not suitable for deployment — agent reasoning is unstable across identical inputs |

In [15]:
print('Running consistency test (3 runs, same conservative profile)...')
print('This may take a few minutes...\n')

consistency = consistency_test(PROFILE_1, model=MODEL_1, n_runs=3, verbose=True)

print(f'\n=== Consistency Results ===')
print(f'Model:             {consistency["model"]}')
print(f'Runs:              {consistency["n_runs"]}')
print(f'Pairwise Jaccard:  {consistency["pairwise_jaccard"]}')
print(f'Average Jaccard:   {consistency["avg_jaccard"]}  (1.0 = identical, 0.0 = completely different)')
print(f'Verdict:           {consistency["verdict"]}')
print(f'\nTickers found in each run:')
for i, tickers in enumerate(consistency['ticker_sets'], 1):
    print(f'  Run {i}: {tickers}')

save_trace(consistency, trace_id='consistency')
print('\nSaved consistency results.')

Running consistency test (3 runs, same conservative profile)...
This may take a few minutes...


[consistency] Run 1/3 ...

[consistency] Run 2/3 ...

[consistency] Run 3/3 ...

=== Consistency Results ===
Model:             gemini-2.5-flash
Runs:              3
Pairwise Jaccard:  [0.429, 0.625, 0.444]
Average Jaccard:   0.499  (1.0 = identical, 0.0 = completely different)
Verdict:           MODERATE

Tickers found in each run:
  Run 1: ['JNJ', 'KO', 'MUST', 'ONLY', 'USD']
  Run 2: ['JNJ', 'KO', 'PG', 'USD', 'VZ']
  Run 3: ['FIRST', 'JNJ', 'KO', 'MUST', 'ONLY', 'PG', 'READ', 'USD']
[trace] Saved -> data\traces\trace_consistency.json

Saved consistency results.


---
## Final Summary Table

In [16]:
print('=== FINAL EVALUATION SUMMARY ===')
if not eval_df.empty:
    summary_cols = [c for c in ['trace', 'model', 'overall', 'reasoning_quality', 'risk_alignment', 'clarity', 'plan_accuracy', 'tokens'] if c in eval_df.columns]
    print(eval_df[summary_cols].to_string(index=False))
    print(f'\nMean overall score: {eval_df["overall"].mean():.2f} / 5.0')
    print(f'Best trace:         {eval_df.loc[eval_df["overall"].idxmax(), "trace"]}')
    print(f'Worst trace:        {eval_df.loc[eval_df["overall"].idxmin(), "trace"]}')
else:
    print('Run the evaluation cells above first.')

=== FINAL EVALUATION SUMMARY ===
                trace            model  overall  reasoning_quality  risk_alignment  clarity  plan_accuracy  tokens
     1 - Conservative gemini-2.5-flash     4.25                  4               5        4              4    8100
       2 - Aggressive gemini-2.5-flash     4.25                  4               5        3              5   13648
         3 - Moderate gemini-2.5-flash     4.50                  5               4        4              5   17254
5a - gemini-2.5-flash gemini-2.5-flash     4.50                  4               5        4              5   13863
     5b - gpt-4o-mini      gpt-4o-mini     4.50                  4               5        4              5   13877

Mean overall score: 4.40 / 5.0
Best trace:         3 - Moderate
Worst trace:        1 - Conservative


---
## Pipeline vs Agent Alignment

The data pipeline (Notebook 1) ranked each ticker by a composite score of news momentum,
sentiment, and fundamentals. The agent (Notebook 2) reasoned independently over the
same raw signals. This section compares the two rankings to see how well the LLM
reasoning aligns with the quantitative pipeline output.

In [17]:
# === Pipeline vs Agent alignment =============================================
if pipeline_df is not None and 'ticker' in pipeline_df.columns and 'trend_score' in pipeline_df.columns:
    pipeline_ranked = pipeline_df.sort_values('trend_score', ascending=False)['ticker'].tolist()

    # Collect all tickers the agent actually bought across the 5 traces
    agent_picks = {}
    for trace_id, label, trace_obj in [
        (1, 'Conservative', trace1),
        (2, 'Aggressive',   trace2),
        (3, 'Moderate',     trace3),
    ]:
        plan_calls = [tc for tc in trace_obj.get('tool_calls', []) if tc['tool'] == 'create_purchase_plan']
        if plan_calls:
            picks = [t for t in plan_calls[-1]['args'].get('tickers', []) if t not in ('CASH', 'USD', 'RISK')]
            agent_picks[label] = picks

    print('=== Pipeline Ranking vs Agent Selections ===\n')
    print(f'Pipeline top-ranked order: {pipeline_ranked}\n')
    for label, picks in agent_picks.items():
        pipeline_positions = [pipeline_ranked.index(t) + 1 if t in pipeline_ranked else "not in pipeline" for t in picks]
        print(f'{label} trace agent picked: {picks}')
        print(f'  Pipeline rank positions: {dict(zip(picks, pipeline_positions))}')
        in_pipeline = [t for t in picks if t in pipeline_ranked]
        print(f'  Overlap with pipeline universe: {len(in_pipeline)}/{len(picks)} tickers')
        print()
else:
    print('Pipeline data not loaded. Run Notebook 1 first and re-run cell 3 of this notebook.')

=== Pipeline Ranking vs Agent Selections ===

Pipeline top-ranked order: ['NVDA', 'AVGO', 'MRVL', 'INTC', 'ORCL', 'MU', 'SNDK', 'AMD', 'HD', 'SDOT', 'CAST', 'SLBT']

Aggressive trace agent picked: ['AMD', 'NVDA']
  Pipeline rank positions: {'AMD': 8, 'NVDA': 1}
  Overlap with pipeline universe: 2/2 tickers

Moderate trace agent picked: ['GOOGL', 'AAPL', 'MSFT']
  Pipeline rank positions: {'GOOGL': 'not in pipeline', 'AAPL': 'not in pipeline', 'MSFT': 'not in pipeline'}
  Overlap with pipeline universe: 0/3 tickers



---
## ROI Analysis Commentary

Based on the model comparison, I think GPT-4o-mini gave the best value for this project. Both Gemini 2.5 Flash and GPT-4o-mini received the same score of 4.5 out of 5, but Gemini cost $0.006161 per report while GPT-4o-mini only cost $0.002559. This means Gemini was about 2.4 times more expensive without giving a better result. GPT-4o-mini also finished the workflow in fewer turns. If the system created 100,000 reports each year, using GPT-4o-mini could save about $360 compared with Gemini. Because the quality was the same in this test, I would choose GPT-4o-mini as the main model.

---
## Backtesting Commentary


The conservative portfolio earned a return of 1.64 percent, while SPY earned 14.23 percent during the same period. This means the portfolio underperformed SPY by 12.59 percent. However, it still passed the basic thresholds because it had a positive return and none of the individual stocks lost more than 30 percent.
One limitation is that the saved trace did not contain a create_purchase_plan tool call. Because of this, the notebook used the fallback stocks JNJ, PG, KO, and WMT instead of the exact stocks from the agent’s report. I think this backtest shows that the function works, but it does not fully measure the actual recommendation made by the agent. In the future, I would backtest the exact stocks and allocation weights from the purchase plan over a longer period.

---
## Overall Agent Evaluation Commentary

Overall, I think the agent performed well for a course prototype. The five reports received an average score of 4.40 out of 5, and all of them passed the evaluation thresholds. The strongest areas were risk alignment and plan accuracy because the agent usually selected stocks that matched each investor’s risk level and created reasonable purchase plans. The moderate trace received the highest score of 4.5. However, the aggressive report received a clarity score of 3, which shows that some of the financial explanations could be easier for beginner investors to understand.

The agent also successfully rejected both off-topic questions without calling unnecessary tools. The consistency score was 0.499, which means the recommendations were only moderately consistent across repeated runs. During human review, I also noticed that the ticker extraction method incorrectly identified words such as MUST, ONLY, READ, and USD as stock symbols. This shows that automated evaluation is helpful, but a human reviewer is still needed to check the traces and find mistakes.

The agent exceeded my expectations in tool calling, risk-based personalization, purchase planning, and rejecting unrelated questions. Its main weaknesses were consistency, evaluation accuracy, and the connection between the different notebook outputs. I learned that building an AI agent is not only about creating a good final answer. The tools, traces, calculations, saved files, and evaluation process also need to work together correctly before the agent can be trusted.